# 15_analyze_mutants


## Python Path Setup
Ensure project-root imports work whether Jupyter starts from repo root or `notebooks/`.


In [ ]:
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
src_root = repo_root / "src"
if src_root.exists() and str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))


## Imports
Load helper functions for mutant table loading, LLM analysis, output export, and thread persistence.


In [ ]:
import importlib
import agentic_protein_design.steps.analyze_mutants as am
am = importlib.reload(am)
from project_config.local_api_keys import OPENAI_API_KEY
from agentic_protein_design.core import apply_notebook_markdown_style, resolve_input_path

default_user_inputs = am.default_user_inputs
default_input_paths = am.default_input_paths
setup_data_root = am.setup_data_root
get_step_processed_dir = am.get_step_processed_dir
init_thread = am.init_thread
load_mutant_table = am.load_mutant_table
load_optional_context = am.load_optional_context
generate_llm_mutant_explanations = am.generate_llm_mutant_explanations
save_mutant_outputs = am.save_mutant_outputs
save_mutant_explanations_csv = am.save_mutant_explanations_csv
save_llm_analysis = am.save_llm_analysis
persist_thread_update = am.persist_thread_update

apply_notebook_markdown_style(font_size_px=14, line_height=1.4)


## API Key Setup
Load the OpenAI key from `project_config/local_api_keys.py` into environment variables for LLM calls.


In [ ]:
if OPENAI_API_KEY and OPENAI_API_KEY != "REPLACE_WITH_YOUR_OPENAI_API_KEY":
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

"OPENAI_API_KEY" in os.environ


## User Inputs
Edit run parameters here: dataset root, thread selection, model settings, and input path(s).


In [ ]:
root_key = "examples"
existing_thread_key = None

user_inputs = {
    "focus_question": (
        "Explain which mutations or mutated positions likely drive changes in activity/selectivity, "
        "and identify likely neutral mutations."
    ),
    "llm_model": "gpt-5.2",
    "llm_temperature": 0.2,
    "llm_max_rows": 500,
    "display_llm_output": True,
    "display_max_height": "640px",
    "context_thread_key": "",  # Optional: e.g., prior binding-pocket or literature thread key
}

input_paths = {
    # relative to selected data_root
    "mutants_csv": "expdata/mutants_to_analyze.csv",
}

# Optional:
# user_inputs = default_user_inputs()


## Setup Runtime Context


In [ ]:
data_root, resolved_dirs = setup_data_root(root_key)
step_processed_dir = get_step_processed_dir(resolved_dirs)
thread, threads_preview = init_thread(root_key, existing_thread_key)
thread_id = thread["thread_id"]
data_root, step_processed_dir, thread_id


## Load Mutant Table


In [ ]:
mutants_csv = resolve_input_path(data_root, input_paths["mutants_csv"])
mutant_df = load_mutant_table(mutants_csv)
mutants_csv, mutant_df.head(10)


## Save Input Snapshot


In [ ]:
out_mutants_snapshot = save_mutant_outputs(mutant_df, step_processed_dir)
out_mutants_snapshot


## Optional Context


In [ ]:
context_thread_key = str(user_inputs.get("context_thread_key", "")).strip() or None
context_result = load_optional_context(context_thread_key)
supplemental_context = str(context_result.get("context_text", ""))
context_thread_key, (supplemental_context[:500] + "...") if supplemental_context else ""


## LLM Mutant Analysis


In [ ]:
explanations_df, llm_json_text = generate_llm_mutant_explanations(
    mutant_df,
    user_inputs,
    supplemental_context=supplemental_context,
)
out_explanations_csv = save_mutant_explanations_csv(explanations_df, step_processed_dir)
llm_analysis_text = (
    "Mutant explanation units (CSV exported):\n\n"
    + (explanations_df.to_markdown(index=False) if not explanations_df.empty else "No rows.")
    + "\n\nRaw LLM JSON:\n```json\n"
    + llm_json_text
    + "\n```"
)
out_llm = save_llm_analysis(llm_analysis_text, step_processed_dir)
out_explanations_csv


## Save Thread Update


In [ ]:
persist_thread_update(
    root_key=root_key,
    thread_id=thread_id,
    user_inputs=user_inputs,
    input_paths=input_paths,
    mutants_snapshot_path=out_mutants_snapshot,
    explanations_csv_path=out_explanations_csv,
    llm_analysis_path=out_llm,
    llm_analysis_text=llm_analysis_text,
    context_thread_key=context_thread_key,
)
print(thread_id)
